# 03 传统分类方法对比

本 Notebook 是实验流程的**第三阶段**，用若干**传统机器学习分类器**在相同数据（第一阶段 PCA15 降维后的中心像元光谱特征）与相同划分（训练/验证/测试 = 24%/6%/70%）上做简单测试，以便与第二阶段的 HybridSN 深度学习模型公平对比。

每个代码块测试一种分类器，输出测试集整体精度 OA、平均精度 AA 与 Cohen's Kappa，最后统一汇总对比。

> 说明：传统方法以**中心像元光谱**为输入（而非 HybridSN 的邻域 patch），这是 HSI 分类中传统方法的常规设置。

In [ ]:
# ==================== 0. 环境配置与路径验证 ====================
import json
import random
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import yaml

matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# ---- 路径 ----
# 项目根目录（实验交付/）：向上查找含 configs/ 与 README.md 的目录，保证无论从何处启动 Jupyter 都能定位
PROJECT_ROOT = Path.cwd().resolve()
for _p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (_p / "configs").is_dir() and (_p / "README.md").is_file():
        PROJECT_ROOT = _p
        break
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "stage3"

# ---- 随机种子 ----
SEED = 1442
random.seed(SEED); np.random.seed(SEED)
print("PROJECT_ROOT =", PROJECT_ROOT)

## 1. 加载第一阶段数据

读取 notebook 01 生成的模型就绪数据，提取**中心像元光谱特征**矩阵 `X` 与标签 `y`，并按第一阶段保存的划分切分训练/验证/测试集。

In [ ]:
# ==================== 1. 加载第一阶段数据 ====================
MANIFEST_PATH = PROJECT_ROOT / "outputs" / "stage1" / "pavia_university" / "stage1_manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print("dataset =", manifest["dataset"], "| selected_route =", manifest["selected_route"])

ARTIFACT_PATH = PROJECT_ROOT / manifest["selected_artifact"]
data = np.load(ARTIFACT_PATH, allow_pickle=False)
cube = data["transformed_cube"]                # (H, W, B) 已标准化+降维立方体
coords = data["coordinates"].astype(np.int32)  # (N, 2)
raw_labels = data["raw_labels"].astype(np.int16)
train_idx = data["train_indices"].astype(np.int64)
val_idx = data["validation_indices"].astype(np.int64)
test_idx = data["test_indices"].astype(np.int64)
num_classes = int(data["num_classes"])

# 中心像元光谱特征（N, B）；标签转 0 基
X_all = cube[coords[:, 0], coords[:, 1], :].astype(np.float32)
y_all = (raw_labels - 1).astype(np.int64)
print(f"特征 X={X_all.shape}  标签 y={y_all.shape}  类别数={num_classes}")

# 按第一阶段保存的划分切分
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val,   y_val   = X_all[val_idx],   y_all[val_idx]
X_test,  y_test  = X_all[test_idx],  y_all[test_idx]
print(f"train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

## 2. 公共评估函数与配置

定义统一的评估函数（OA / AA / Kappa / 混淆矩阵），并读取阶段三配置中的分类器超参数。

In [ ]:
# ==================== 2. 公共评估函数与配置 ====================
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score

def evaluate(y_true, y_pred):
    """计算整体精度 OA、平均精度 AA、Cohen's Kappa 与混淆矩阵。"""
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    oa = accuracy_score(y_true, y_pred)
    per_class = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
    aa = per_class.mean()
    kappa = cohen_kappa_score(y_true, y_pred)
    return {"oa": float(oa), "aa": float(aa), "kappa": float(kappa),
            "cm": cm.tolist(), "per_class": per_class.tolist()}

# 读取阶段三配置（分类器超参数集中在 configs/stage3_traditional/*.yaml）
CONFIG_PATH = PROJECT_ROOT / "configs" / "stage3_traditional" / "pavia_traditional.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
results = {}   # 汇总每种方法的结果

## 3. SVM（支持向量机）

RBF 核 SVM，超参数 `C` 与 `gamma` 可在配置中调整。

In [ ]:
# ==================== 3. SVM（支持向量机） ====================
from sklearn.svm import SVC
svm_cfg = cfg["classifiers"]["svm"]
svm = SVC(kernel=svm_cfg["kernel"],       # 核函数：rbf
          C=svm_cfg["C"],                 # 正则化系数（越大越易过拟合，可调）
          gamma=svm_cfg["gamma"],         # 核系数：scale = 1/(n_features*var)
          random_state=SEED)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
results["SVM"] = evaluate(y_test, svm_pred)
r = results["SVM"]
print(f"SVM  OA={r['oa']*100:.2f}%  AA={r['aa']*100:.2f}%  Kappa={r['kappa']*100:.2f}%")

## 4. KNN（K 近邻）

距离加权的 K 近邻分类，近邻数 `n_neighbors` 可调。

In [ ]:
# ==================== 4. KNN（K 近邻） ====================
from sklearn.neighbors import KNeighborsClassifier
knn_cfg = cfg["classifiers"]["knn"]
knn = KNeighborsClassifier(n_neighbors=knn_cfg["n_neighbors"],  # 近邻数（可调）
                           weights=knn_cfg["weights"])           # 距离加权
knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)
results["KNN"] = evaluate(y_test, knn_pred)
r = results["KNN"]
print(f"KNN  OA={r['oa']*100:.2f}%  AA={r['aa']*100:.2f}%  Kappa={r['kappa']*100:.2f}%")

## 5. LDA（线性判别分析）

线性判别分析直接作为分类器，无需额外超参数。

In [ ]:
# ==================== 5. LDA（线性判别分析） ====================
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda_clf = LinearDiscriminantAnalysis()
lda_clf.fit(X_train, y_train)
lda_pred = lda_clf.predict(X_test)
results["LDA"] = evaluate(y_test, lda_pred)
r = results["LDA"]
print(f"LDA  OA={r['oa']*100:.2f}%  AA={r['aa']*100:.2f}%  Kappa={r['kappa']*100:.2f}%")

## 6. 随机森林（Random Forest）

集成多棵决策树，树的数量 `n_estimators` 可调。

In [ ]:
# ==================== 6. 随机森林 ====================
from sklearn.ensemble import RandomForestClassifier
rf_cfg = cfg["classifiers"]["random_forest"]
rf = RandomForestClassifier(n_estimators=rf_cfg["n_estimators"],   # 树的数量（可调）
                            max_depth=rf_cfg["max_depth"],         # 树的最大深度（None=不限制）
                            max_features=rf_cfg["max_features"],   # 每次分裂考虑的特征数
                            random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results["RandomForest"] = evaluate(y_test, rf_pred)
r = results["RandomForest"]
print(f"RandomForest  OA={r['oa']*100:.2f}%  AA={r['aa']*100:.2f}%  Kappa={r['kappa']*100:.2f}%")

## 7. XGBoost（梯度提升树）

梯度提升决策树，提升轮数 `n_estimators`、学习率 `learning_rate` 等可调。

In [ ]:
# ==================== 7. XGBoost（梯度提升树） ====================
from xgboost import XGBClassifier
xgb_cfg = cfg["classifiers"]["xgboost"]
xgb = XGBClassifier(n_estimators=xgb_cfg["n_estimators"],         # 提升轮数（可调）
                    max_depth=xgb_cfg["max_depth"],               # 树深度
                    learning_rate=xgb_cfg["learning_rate"],       # 学习率
                    subsample=xgb_cfg["subsample"],               # 样本采样比例
                    colsample_bytree=xgb_cfg["colsample_bytree"], # 特征采样比例
                    eval_metric="mlogloss",
                    random_state=SEED, n_jobs=-1)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
results["XGBoost"] = evaluate(y_test, xgb_pred)
r = results["XGBoost"]
print(f"XGBoost  OA={r['oa']*100:.2f}%  AA={r['aa']*100:.2f}%  Kappa={r['kappa']*100:.2f}%")

## 8. 方法汇总对比

汇总所有传统方法的测试集指标，保存结果并绘制对比柱状图。

In [ ]:
# ==================== 8. 方法汇总对比 ====================
EXP_DIR = OUTPUT_DIR / cfg["experiment"]["name"]
EXP_DIR.mkdir(parents=True, exist_ok=True)

# 保存汇总结果（供报告使用）
(EXP_DIR / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印对比表
print(f"{'方法':16s} {'OA (%)':>8s} {'AA (%)':>8s} {'Kappa (%)':>10s}")
for name, r in results.items():
    print(f"{name:16s} {r['oa']*100:8.2f} {r['aa']*100:8.2f} {r['kappa']*100:10.2f}")

# 对比柱状图
names = list(results.keys())
oa = [results[n]["oa"]*100 for n in names]
aa = [results[n]["aa"]*100 for n in names]
kappa = [results[n]["kappa"]*100 for n in names]
x = np.arange(len(names)); width = 0.25
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(x - width, oa, width, label="OA")
ax.bar(x, aa, width, label="AA")
ax.bar(x + width, kappa, width, label="Kappa")
for i in range(len(names)):
    ax.text(x[i]-width, oa[i], f"{oa[i]:.1f}", ha="center", va="bottom", fontsize=7)
    ax.text(x[i], aa[i], f"{aa[i]:.1f}", ha="center", va="bottom", fontsize=7)
    ax.text(x[i]+width, kappa[i], f"{kappa[i]:.1f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha="right")
ax.set_ylabel("得分 (%)"); ax.set_title("传统分类方法对比（Pavia University，PCA15 中心像元光谱）")
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_ylim(72, 96)  # 纵轴截断（不从 0 开始），放大各方法之间的差异
plt.tight_layout()
plt.savefig(EXP_DIR / "methods_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("结果已保存 ->", EXP_DIR.relative_to(PROJECT_ROOT))